# AI Project 1

## Spring & Summer of 2026
## Dr. Tanghatari

## Group 7
- Mohammad Amin Haghjoo - 403110585
- Pouya Astaraki - 403110477

# 1.1 Packages and dependencies

All packages we used are imported here

In [9]:
import numpy as np
import pandas as pd
import os
from collections import Counter
from typing import List, Tuple


# 1.2 Dataset loading

The WIDSM dataset is loaded in this cell.

In [14]:
def load_wisdm(file_path: str) -> pd.DataFrame:
    """
    Load the WISDM activity recognition dataset from a text file.

    Handles:
      - Multiple data rows on a single line (separated by ';')
      - Lines ending with ',;' instead of just ';' (A dataset format bug)
      - Empty lines and extra whitespace

    Parameters
    ----------
    file_path : str
        Path to the .txt file.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        ['userID', 'activityLabel', 'timestamp', 'ax', 'ay', 'az']
        - userID       : int
        - activityLabel: str
        - timestamp    : int (epoch time in milliseconds)
        - ax, ay, az   : float
    """
    records: List[Tuple] = []

    i: int = 0
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Split by semicolon to get individual records.
            # This handles both ';' and ',;' because the trailing comma will
            # be part of the last record string before the semicolon.
            parts = line.split(';')

            for part in parts:
                part = part.strip()
                if not part:
                    continue

                # Remove trailing commas (caused by ',;' endings)
                if part.endswith(','):
                    part = part[:-1]

                # Split by comma to get fields
                fields = part.split(',')

                # Expect exactly 6 fields
                if len(fields) != 6:
                    print(f"error at line: {i + 1}")
                    raise ValueError('WISDM activity recognition dataset must have 6 fields')


                user_id = int(fields[0].strip())
                activity = fields[1].strip()
                timestamp = int(fields[2].strip())
                ax = float(fields[3].strip())
                ay = float(fields[4].strip())
                az = float(fields[5].strip())
                records.append((user_id, activity, timestamp, ax, ay, az))
                i = i + 1

    # Build DataFrame
    df = pd.DataFrame(
        records,
        columns=['userID', 'activityLabel', 'timestamp', 'ax', 'ay', 'az']
    )

    # Convert data types explicitly (Pandas will infer, but we enforce)
    df = df.astype({
        'userID': 'int64',
        'activityLabel': 'string',
        'timestamp': 'int64',
        'ax': 'float64',
        'ay': 'float64',
        'az': 'float64'
    })

    return df

df = load_wisdm('WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt')

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 1098208 entries, 0 to 1098207
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   userID         1098208 non-null  int64  
 1   activityLabel  1098208 non-null  string 
 2   timestamp      1098208 non-null  int64  
 3   ax             1098208 non-null  float64
 4   ay             1098208 non-null  float64
 5   az             1098208 non-null  float64
dtypes: float64(3), int64(2), string(1)
memory usage: 50.3 MB
None
   userID activityLabel       timestamp        ax         ay        az
0      33       Jogging  49105962326000 -0.694638  12.680544  0.503953
1      33       Jogging  49106062271000  5.012288  11.264028  0.953424
2      33       Jogging  49106112167000  4.903325  10.882658 -0.081722
3      33       Jogging  49106222305000 -0.612916  18.496431  3.023717
4      33       Jogging  49106332290000 -1.184970  12.108489  7.205164


# 1.3 Deciding on the training/test split

As specified by the project spec, we will choose some users for training and others for testing. To do so, we will first look at how many entries belong to each user.

In [15]:
counts = df['userID'].value_counts().reindex(range(1, 37), fill_value=0).sort_index()

# Display the result
print("Entries per userID (1 to 36):")
print(counts)

# Optional: show total entries and number of unique users present
print(f"\nTotal entries: {len(df)}")
print(f"Unique userIDs in dataset: {df['userID'].nunique()}")

# Take user IDs 1 to 27 for training set.

Entries per userID (1 to 36):
userID
1     29978
2     23525
3     35161
4     11371
5     28510
6     29704
7     30967
8     41188
9     12923
10    34883
11    31658
12    32641
13    37093
14    38192
15    28155
16    20470
17    22020
18    32812
19    45383
20    56632
21    35436
22    22310
23    25673
24    25736
25    13468
26    32578
27    34959
28    21358
29    38245
30    25335
31    44282
32    35506
33    29453
34    35948
35    22394
36    32261
Name: count, dtype: int64

Total entries: 1098208
Unique userIDs in dataset: 36


# 1.4 Performing dataset split and sequence creation.

Users 1 to 27 are used for the training set.  (Approx. 74\% of the dataset.) Users 28 to 31 are used for the validation set (Approx. 12\% of the dataset.), and 32 to 36 are used for the test set. (Approx. 14\% of the dataset.)

In [ ]:
def create_rnn_sequences(df: pd.DataFrame, T: int, training_cutoff=27, validation_cutoff=31) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Create sliding-window sequences for RNN training.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns ['userID', 'activityLabel', 'timestamp', 'ax', 'ay', 'az'].
    T : int
        Sequence length (number of time steps per sample).
    training_cutoff : int
        The cutoff point between training set/validation set.
    validation_cutoff : int
        The cutoff point between validation set/test set.

    Returns
    -------
    X_train : np.ndarray, shape (n_samples_train, T, 3)
        Feature sequences (ax, ay, az) for each window.
    y_train : np.ndarray, shape (n_samples_train,)
        Majority-vote activity label for each sequence (strings).
    X_validation : np.ndarray, shape (n_samples_validation, T, 3)
        Feature sequences (ax, ay, az) for each window.
    y_validation : np.ndarray, shape (n_samples_validation,)
        Majority-vote activity label for each sequence (strings).
    X_test : np.ndarray, shape (n_samples_test, T, 3)
        Feature sequences (ax, ay, az) for each window.
    y_test : np.ndarray, shape (n_samples_test,)
        Majority-vote activity label for each sequence (strings).
    """
    X_list_train = []
    y_list_train = []

    X_list_test = []
    y_list_test = []

    X_list_validation = []
    y_list_validation = []

    # Group by user to keep sequences person‑specific
    for user_id, group in df.groupby('userID', sort=False):

        # Sort by time within each user
        group = group.sort_values('timestamp')

        # Extract features and labels as numpy arrays
        features = group[['ax', 'ay', 'az']].values  # shape (n_rows, 3)
        labels = group['activityLabel'].values       # shape (n_rows,)

        n_rows = len(features)
        if n_rows < T:
            # Not enough data for a single sequence – skip this user
            continue

        # Slide a window of length T with step size 1
        for start in range(n_rows - T + 1):
            end = start + T
            seq_X = features[start:end]  # shape (T, 3)

            # Majority vote for the label
            seq_labels = labels[start:end]
            # Counter.most_common(1) returns [(label, count)] – if tie, it picks the one
            # that appears first in the original order among the tied labels.
            majority_label = Counter(seq_labels).most_common(1)[0][0]

            if user_id <= training_cutoff:
                X_list_train.append(seq_X)
                y_list_train.append(majority_label)
            elif user_id <= validation_cutoff:
                X_list_validation.append(seq_X)
                y_list_validation.append(majority_label)
            else:
                X_list_test.append(seq_X)
                y_list_test.append(majority_label)

    X_train = np.array(X_list_train, dtype=np.float32)
    y_train = np.array(y_list_train, dtype=object)  # keep as strings

    X_validation = np.array(X_list_validation, dtype=np.float32)
    y_validation = np.array(y_list_validation, dtype=object)

    X_test = np.array(X_list_test, dtype=np.float32)
    y_test = np.array(y_list_test, dtype=object)


    return X_train, y_train, X_validation, y_validation, X_test, y_test


# ================= Example Usage =================
# if __name__ == "__main__":
#     # Load the dataset (using the function from the previous step)
#     df = load_wisdm("WISDM_ar_v1.1_raw.txt")
#
#     # Choose sequence length
#     T = 5  # hyperparameter
#
#     X, y = create_rnn_sequences(df, T)
#
#     print(f"Total sequences generated: {X.shape[0]}")
#     print(f"Each sequence has {X.shape[1]} time steps and {X.shape[2]} features (ax, ay, az).")
#     print(f"Labels shape: {y.shape}")
#
#     # Example: first sequence and its label
#     print("\nFirst sequence (first 3 time steps):")
#     print(X[0][:3])
#     print(f"Label for first sequence: {y[0]}")
#
#     # If you need numeric labels for classification:
#     from sklearn.preprocessing import LabelEncoder
#     le = LabelEncoder()
#     y_encoded = le.fit_transform(y)
#     print(f"\nEncoded labels: {y_encoded[:10]}")
#
#     # Now X (float32) and y_encoded (int) are ready for an RNN like LSTM/GRU.

# 1.6 Normalizing the dataset

For each sequence entry and all sequences, the mean and standard deviation are calculated and the acceleration components are normalized.

$$ \mathbf{a_{\text{norm}}} = (\frac{a_x - \mu_x}{\sigma_x}, \frac{a_y - \mu_y}{\sigma_y}, \frac{a_z - \mu_z}{\sigma_z}) $$

In [ ]:
def normalize_components(X: np.ndarray) -> np.ndarray:
    """
    Normalize each acceleration component (axis) by subtracting the global mean
    and dividing by the global standard deviation.

    The statistics (mean, std) are computed over all time steps and all samples
    pooled together, independently for each of the 3 axes.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, T, 3)
        Input acceleration sequences.

    Returns
    -------
    X_norm : np.ndarray, shape (n_samples, T, 3)
        Normalized sequences with the same shape.
    """
    # Flatten to (n_samples * T, 3) to compute global per‑axis statistics
    original_shape = X.shape
    X_flat = X.reshape(-1, 3)   # shape: (N*T, 3)

    # Compute mean and std for each axis
    mean = np.mean(X_flat, axis=0)   # shape: (3,)
    std = np.std(X_flat, axis=0)     # shape: (3,)

    # Safeguard: if an axis has zero variance, avoid division by zero.
    # Since (x - mean) will be 0 for all values, setting std to 1 keeps the result 0.
    std = np.where(std == 0, 1.0, std)

    # Normalize and reshape back
    X_norm_flat = (X_flat - mean) / std
    X_norm = X_norm_flat.reshape(original_shape)

    return X_norm


# # ================= Example Usage =================
# if __name__ == "__main__":
#     # Assuming you already have X from the previous step:
#     # X, y = create_rnn_sequences(df, T)
#
#     # For demonstration, let's create dummy data:
#     # 100 samples, each of length T=5, with 3 features
#     X_dummy = np.random.randn(100, 5, 3) * 2 + 1.5
#
#     X_norm = normalize_components(X_dummy)
#
#     print("Original shape:", X_dummy.shape)
#     print("Normalized shape:", X_norm.shape)
#
#     # Verify that each axis has mean ≈ 0 and std ≈ 1 across all time steps/samples
#     # Pool all time steps and samples for each axis:
#     print("\n--- Verification ---")
#     # Reshape to (N*T, 3) and compute stats
#     pooled = X_norm.reshape(-1, 3)
#     for i, axis in enumerate(['ax', 'ay', 'az']):
#         print(f"{axis}: mean = {pooled[:, i].mean():.6f}, std = {pooled[:, i].std():.6f}")

# 2.1 RNN from scratch

We have implemented a RNN class with adjustable parameters to be used and trained.

In [1]:
class RNN_scratch:

    def __init__(self, hidden_size: int, input_size=3, output_size=6):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.hs = []
        self.probs = None
        self.h_T = None

        # He initialization
        self.W_xh = np.random.randn(self.input_size, self.hidden_size) * np.sqrt(2 / self.input_size)
        self.W_hh = np.random.randn(self.hidden_size, self.hidden_size) * np.sqrt(2 / self.hidden_size)
        self.W_hy = np.random.randn(self.hidden_size, self.output_size) * np.sqrt(2 / self.hidden_size)
        self.b_h = np.zeros((1, self.hidden_size))
        self.b_y = np.zeros((1, self.output_size))

    def forward(self, X: np.ndarray, y_true: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Forward propagation for the RNN model.
        :param X: A numpy 3D array of shape (batch_size, T, 3)
        :param y_true: The correct label of the input sequences, shape (batch_size,) containing the correct index.
        :return: Returns a numpy 2D array of shape (batch_size, output_size) which are the output probabilities of the RNN model.
        """
        y_true = y_true.flatten()
        self.hs = []
        batch_size, T, _ = X.shape

        h = np.zeros((batch_size, self.hidden_size)) # h_0 = 0

        for t in range(T):
            X_t = X[:, t, :]
            h = np.tanh(
                X_t @ self.W_xh + h @ self.W_hh + self.b_h
            )
            self.hs.append(h)
        self.h_T = h
        logits = h @ self.W_hy + self.b_y
        max_vals = np.max(logits, axis=1, keepdims=True)  # (B, 1)
        exp_logits = np.exp(logits - max_vals)            # (B, C)
        self.probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

        # Select the probability of the true class for each sample
        # Using advanced indexing: probs[range(B), y_true]
        correct_probs = self.probs[np.arange(batch_size), y_true]
        # Add a tiny epsilon to avoid log(0)
        eps = 1e-15
        loss = -np.mean(np.log(correct_probs + eps))
        return self.probs, loss

    def backward(self, X: np.ndarray, y_true: np.ndarray):
        """
        Backward propagation for the RNN model.
        """
        y_true = y_true.flatten()
        batch_size, T, _ = X.shape

        y_one_hot = np.zeros((batch_size, self.output_size))
        y_one_hot[np.arange(batch_size), y_true] = 1.0

        d_logits = self.probs - y_one_hot
        d_W_hy = (self.h_T.T @ d_logits) / batch_size
        d_b_y = np.sum(d_logits, axis=0, keepdims=True) / batch_size

        d_W_hh = np.zeros_like(self.W_hh)
        d_W_xh = np.zeros_like(self.W_xh)
        d_b_h = np.zeros_like(self.b_h)

        d_h = d_logits @ self.W_hy.T
        d_z = np.zeros_like(d_h)
        for t in reversed(range(T)):
            d_z = d_h * (1.0 - self.hs[t] ** 2)
            d_h = d_z @ self.W_hh.T

            d_W_xh += X[:, t, :].T @ d_z / batch_size

            if t == 0:
                h_last = np.zeros_like(d_h)
            else:
                h_last = self.hs[t - 1]

            d_W_hh += h_last.T @ d_z / batch_size

            d_b_h += np.sum(d_z, axis=0, keepdims=True) / batch_size




_IncompleteInputError: incomplete input (2897527717.py, line 68)